In [1]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents.middleware import ToolCallLimitMiddleware, AgentMiddleware, ModelRequest, wrap_tool_call
from langchain.agents import create_agent

from typing import Annotated, Sequence, TypedDict,Literal, List, Dict, Tuple, Union
import functools
import os
import threading

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    ToolMessage,
    AIMessage,
)
from langchain_anthropic import ChatAnthropic
# from langchain_openai import AzureChatOpenAI
# from langchain_deepseek import ChatDeepSeek

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, create_react_agent
from pydantic import BaseModel, Field

from src.tools import *
from src.prompt import dft_agent_prompt,hpc_agent_prompt,supervisor_prompt, oer_agent_prompt
from src import var

members = ["OER_Agent"]

class myStep(BaseModel):
    """Step in the plan."""

    step: str = Field(description="Step to perform.")
    agent: str = Field(
        description=f"Agent to perform the step. Should be one of {members}."
    )

class Plan(BaseModel):
    """Plan to follow in future"""

    steps: List[myStep] = Field(
        description=f"""
        Steps to follow in future. Each step is a tuple of (step, agent). agent can only be chosen from {members}.
        """
        # description="different steps to follow, should be in sorted order"
        # description="""different steps to follow (first element of the Tuple), and the agent in charge for each step (second element of the Tuple),
        # should be in sorted order by the order of execution"""
    )
    

class Response(BaseModel):
    """End everything and response to the user."""

    response: str


class Act(BaseModel):
    """Action to perform."""

    action: Union[Plan, Response] = Field(
        description="Action to perform. If you need to further use tools to get the answer, use Plan."
        "If you want to end the conversation, use Response."
        # "DO NOT use response unless absolutly necessary."
    )
    
class wokerResponse(BaseModel):
    """Response from the worker agent."""

    answer: str = Field(
        description="a short summary of the answer to the question or task."
    )
    
    summary: str = Field(
        description="""what have you done + what did you note down? i.e. I did xxx, and got xxx. I did xxx, and found xxx ..... In the end, I answered xxx/finished xxx/failed xxx/... I have noted down xxx, xxx, and xxx on CANVAS"""
    )

class PlanExecute(TypedDict):
    inputs: str
    plan: List[myStep]
    past_steps: List[myStep]
    response: str
    next: str

class DisableParallelToolCallsMiddleware(AgentMiddleware):
    
    def wrap_model_call(self, request, handler):
        request.model_settings["parallel_tool_calls"] = False
        return handler(request)
    
    async def awrap_model_call(self, request, handler):
        request.model_settings["parallel_tool_calls"] = False
        return await handler(request)

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Only handle errors that occur during tool execution due to invalid inputs
        # that pass schema validation but fail at runtime (e.g., invalid SQL syntax).
        # Do NOT handle:
        # - Network failures (use tool retry middleware instead)
        # - Incorrect tool implementation errors (should bubble up)
        # - Schema mismatch errors (already auto-handled by the framework)
        #
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

def print_stream(s):
    if "messages" not in s:
        print("#################")
        if var.my_SAVE_DIALOGUE:
            with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                f.write("#################\n")
        print(s)
        if var.my_SAVE_DIALOGUE:
            with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                f.write(repr(s))
                f.write("\n")
    else:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
            if var.my_SAVE_DIALOGUE:
                with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                    f.write(repr(message))
                    f.write("\n")
        else:
            if hasattr(message, 'usage_metadata'):
                var.TOKEN_USAGE.append(message.usage_metadata)
                print(f"input_tokens: {message.usage_metadata['input_tokens']}, output_tokens: {message.usage_metadata['output_tokens']}")
                if var.my_SAVE_DIALOGUE:
                    with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                        f.write(f"input_tokens: {message.usage_metadata['input_tokens']}, output_tokens: {message.usage_metadata['output_tokens']}\n")
            message.pretty_print()
            if var.my_SAVE_DIALOGUE:
                with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                    f.write(message.pretty_repr())
                    f.write("\n")
    print()
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write("\n")

/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/autocat/data/lattice_parameters/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
MACE imported successfully


In [2]:
def supervisor_chain_node(state, agent, name):
    
    # read "status.txt" in the working directory
    with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
        status = f.read()
    while status == "stop":
        print(f"Calculation pause, supervisor is waiting. cwd: {var.my_WORKING_DIRECTORY}")
        # wait for 5 second
        time.sleep(5)
        with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
            status = f.read()
    
    print(f"supervisor is processing!!!!!")
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"supervisor is processing!!!!!\n")

    print(state)
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(str(state))
            f.write("\n")
            
    plan = state["plan"]
    plan_str = "\n".join(f"{i+1}. {step.step}" for i, step in enumerate(plan))
    # task_formatted = f"""For the following plan:
    # {plan_str}\n\nYou are tasked with executing step {1}, {task}."""
    old_tasks_string = "\n".join(f"{i+1}. {step.agent}: {step.step}" for i, step in enumerate(state["past_steps"]))
    
    supervisorMessage =  f"""
Your available agents are: {members}.

The overall goal is: {state['inputs']}. 

the current plan is:
{plan_str}

this is what has been done:
{old_tasks_string}

Please update the plan accordingly.
    """
        
    for agent_response in agent.stream(
        {"messages": [("user", supervisorMessage)]},  {"configurable": {"thread_id": "1"}, "recursion_limit": 1000}
    ):
        # set agent_response to be the value of the first key of the dictionary
        agent_response = next(iter(agent_response.values()))
        print_stream(agent_response)

    # output = agent.invoke(
    #     {"messages": [("user", supervisorMessage)]},  {"configurable": {"thread_id": "1"}, "recursion_limit": 1000}
    #     )
    
    agent_response = agent_response['structured_response']
    if isinstance(agent_response.action, Response):
        return {"response": agent_response.action.response, "next": "FINISH"}
    # elif isinstance(output.action, Response):
    #     return {"response": "Plan is not finished! Do not use response!", "next": "Supervisor"}
    else:
        plan_str = "\n".join(f"{i+1}. {step.step}" for i, step in enumerate(agent_response.action.steps))
        print(plan_str)
        if var.my_SAVE_DIALOGUE:
            with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                f.write(plan_str)
                f.write("\n")
        return {"plan": agent_response.action.steps, "next": agent_response.action.steps[0].agent}

In [3]:
def worker_agent_node(state, agent, name, past_steps_list):
    # read "status.txt" in the working directory
    with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
        status = f.read()
    while status == "stop":
        print(f"Calculation pause, {name} Agent is waiting. cwd: {var.my_WORKING_DIRECTORY}")
        # wait for 5 second
        time.sleep(5)
        with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
            status = f.read()
    
    print(f"Agent {name} is processing!!!!!")
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"Agent {name} is processing!!!!!\n")
        
    plan = state["plan"]
    plan_str = "\n".join(f"{i+1}. {step.step}" for i, step in enumerate(plan))
    # print(plan_str)
    # if var.my_SAVE_DIALOGUE:
    #     with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
    #         f.write(plan_str)
    #         f.write("\n")
    task = plan[0]
#     task_formatted = f"""For the following plan:
# {plan_str}\n\nYou are tasked with executing step {1}, {task}."""
    old_tasks_string = "\n".join(f"{i+1}. {step.agent}: {step.step}" for i, step in enumerate(past_steps_list))
    task_formatted = f"""
Here are what has been done so far:
{old_tasks_string}

Here is the overall objective:
{state["inputs"]}

Now, you are tasked with: {task}. Please only do this task! Do not do anything else! Please note down important information on CANVAS before you end.
"""
    
    print(task_formatted)
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(task_formatted)
            f.write("\n")
    print(f"Agent {name} is processing!!!!!")
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"Agent {name} is processing!!!!!\n")
    
    
    for agent_response in agent.stream(
        {"messages": [("user", task_formatted)]},  {"configurable": {"thread_id": "1"}, "recursion_limit": 1000}
    ):
        # set agent_response to be the value of the first key of the dictionary
        agent_response = next(iter(agent_response.values()))
        print_stream(agent_response)
    
    # agent_response = agent.invoke(
    #     {"messages": [("user", task_formatted)]},  {"configurable": {"thread_id": "1"}}
    # )
    structured_response = agent_response['structured_response']
    
    
    # past_steps_list.append((task, agent_response["messages"][-1].content))
    past_steps_list.append(myStep(step=structured_response.summary, agent=name))
    
    print_stream(structured_response.summary)
    
    return {
        "past_steps": past_steps_list,
    }
    
def whos_next(state):
    return state["next"]

In [5]:
def create_planning_graph(config: dict) -> StateGraph:
    # create a file named status.txt in the working directory
    WORKING_DIRECTORY = var.my_WORKING_DIRECTORY
    with open(f"{WORKING_DIRECTORY}/status.txt", "w") as f:
        f.write("run")
    
    # Define the model
    # llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0)
    # workerllm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0, tool_choice="auto")
    llm = ChatAnthropic(model="claude-sonnet-4-5-20250929", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0)
    workerllm = ChatAnthropic(model="claude-sonnet-4-5-20250929", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0, tool_choice="auto")
    # workerllm = ChatAnthropic(model="claude-3-5-sonnet-20241022", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0)
    # llm = AzureChatOpenAI(model="gpt-4o", api_version="2024-08-01-preview", api_key=config["OpenAI_API_KEY"], azure_endpoint = config["OpenAI_BASE_URL"])
    # workerllm = AzureChatOpenAI(model="gpt-4o", api_version="2024-08-01-preview", api_key=config["OpenAI_API_KEY"], azure_endpoint = config["OpenAI_BASE_URL"], model_kwargs={'parallel_tool_calls': False})
    # llm = ChatDeepSeek(model_name=config["DeepSeek_MDL"], api_key=config['DeepSeek_API_KEY'], api_base=config['DeepSeek_BASE_URL'], temperature=0.0)
    
    var.my_WORKING_DIRECTORY = var.my_WORKING_DIRECTORY
    
    if not eval(config["SAVE_DIALOGUE"]):
        var.my_SAVE_DIALOGUE = False
    
    
    
    # System Supervisor with tool bind and with_structured_output
    
    # supervisor_chain = supervisor_prompt | llm.bind_tools(supervisor_tools).with_structured_output(Act)
    # supervisor_agent = functools.partial(supervisor_chain_node, chain=supervisor_chain, name="Supervisor")
    
    
        
    
    # def supervisor_agent(state):
    #     print("Supervisor!!!!!!!!!")
    #     supervisor_chain = (
    #         prompt
    #         | llm.with_structured_output(routeResponse)
    #     )
    #     return supervisor_chain.invoke(state)
    
    ## Memory Saver
    memory = MemorySaver()

    PAST_STEPS = []
    myCANVAS = {}
    
    supervisor_tools = [
        inspect_my_canvas,
        read_my_canvas,
        inspect_explog
        ]
    
    supervisor_agent = create_agent(
        model=llm,
        tools=supervisor_tools, 
        system_prompt=supervisor_prompt,
        # Structured output via ToolStrategy (tool-calling fallback)
        response_format=ToolStrategy(Act),  # Or ProviderStrategy for native models
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    
    # supervisor_agent = create_react_agent(llm, tools=supervisor_tools,
    #                                prompt=supervisor_prompt, response_format=Act)   
    supervisor_node = functools.partial(supervisor_chain_node, agent=supervisor_agent, name="Supervisor_Agent")
    
    ### DFT Agent
    dft_tools = [
        inspect_my_canvas,
        write_my_canvas,
        read_my_canvas,
        calculate_formation_E,
        generateSurface_and_getPossibleSite,
        generate_myAdsorbate,
        add_myAdsorbate,
        init_structure_data,
        find_pseudopotential,
        write_QE_script_w_ASE,
        calculate_lc,
        generate_convergence_test,
        get_kspacing_ecutwfc,
        generate_eos_test,
        read_energy_from_output,
        get_convergence_suggestions,
        analyze_BEEF_result
        ]
    # dft_agent = create_react_agent(workerllm, tools=dft_tools,
    #                                prompt="You are a DFT expert")   
    dft_agent = create_agent(
        model=workerllm,
        tools=dft_tools,
        system_prompt=dft_agent_prompt,
        response_format=ToolStrategy(wokerResponse),
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    dft_node = functools.partial(worker_agent_node, agent=dft_agent, name="DFT_Agent", past_steps_list=PAST_STEPS)

    
    oer_tools = [
        inspect_explog,
        inspect_my_canvas,
        write_my_canvas,
        read_my_canvas,
        OER_data_analasis_v2,
        read_df,
        arXiv_search,
        enter_candidate_in_log,
        submit_dft_job,
        get_terminations_ranking,
        list_adsorption_sites,
        read_explog,
        get_top_k_candidates,
        extract_df
        ]
    # oer_agent = create_react_agent(workerllm, tools=oer_tools,
    #                                prompt=oer_agent_prompt)
    oer_agent = create_agent(
        model=workerllm,
        tools=oer_tools,
        system_prompt=oer_agent_prompt,
        response_format=ToolStrategy(wokerResponse),
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    oer_node = functools.partial(worker_agent_node, agent=oer_agent, name="OER_Agent", past_steps_list=PAST_STEPS)

    ### HPC Agent
    # hpc_tools = [read_script, submit_and_monitor_job, read_energy_from_output]
    hpc_tools = [
        inspect_my_canvas,
        write_my_canvas,
        read_my_canvas,
        submit_and_monitor_job,
        add_resource_suggestion
        ]

    # hpc_agent = create_react_agent(workerllm, tools=hpc_tools,
    #                                prompt=hpc_agent_prompt)
    hpc_agent = create_agent(
        model=workerllm,
        tools=hpc_tools,
        system_prompt=hpc_agent_prompt,
        response_format=ToolStrategy(wokerResponse),
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    hpc_node = functools.partial(worker_agent_node, agent=hpc_agent, name="HPC_Agent", past_steps_list=PAST_STEPS)
    
    ### MD Agent
    # md_tools = [
    #     find_classical_potential,
    #     init_structure_data,
    #     write_LAMMPS_script
    # ]
    
    # md_agent = create_react_agent(llm, tools=md_tools,
    #                               state_modifier=md_agent_prompt)
    
    # md_node = functools.partial(worker_agent_node, agent=md_agent, name="MD_Agent", past_steps_list=PAST_STEPS)

    # save_graph_to_file(dft_agent, config['working_directory'], "dft_agent")
    


    # Create the graph
    graph = StateGraph(PlanExecute)
    # graph.add_node("DFT_Agent", dft_node)
    # graph.add_node("HPC_Agent", hpc_node)
    graph.add_node("OER_Agent", oer_node)
    # graph.add_node("MD_Agent", md_node)
    # graph.add_node("CSS_Agent", css_node)

    graph.add_node("Supervisor", supervisor_node)
    
    for member in members:
    # We want our workers to ALWAYS "report back" to the supervisor when done
        graph.add_edge(member, "Supervisor")
    # The supervisor populates the "next" field in the graph state
    # which routes to a node or finishes
    conditional_map = {k: k for k in members}
    conditional_map["FINISH"] = END
    conditional_map["Supervisor"] = "Supervisor" 
    graph.add_conditional_edges("Supervisor", whos_next, conditional_map)
    graph.add_edge(START, "Supervisor") 
    # checkpointer = InMemorySaver()
    # return graph.compile(checkpointer=checkpointer)
    return graph.compile()

In [6]:
userMessage_6 = "You are going to calculate the lattice constant for BCC Li through DFT, the experiment value is 3.451, use this to create the initial structure."
userMessage_7 = "You are going to generat a Pt surface structure with 2x2x4 supercell, then do a convergence test, use maximum ecutwfc = 160. Get the optimal kspacing and ecutwfc."
userMessage_8 = """Please generate intial structures required to calculate CO adsorbtion on Pt(111) surface with 1/4 coverage (2x2x4 supercell), and calculate the adsorbtion energy."""
userMessage_9 = """
Please find out the most perfered adsorbtion site and adsorbate orientation (up or down) for CO adsorbtion on Pt(111) surface with 1/4 coverage (2x2x4 supercell).
"""
userMessage_10 = """please find the adsorption energy difference between the most favorable configurations (different adsorbate orientations 0, 90, 180) at fcc site and
most favorable configuration (different adsorbate orientations 0, 90, 180) at ontop site for CO on Pt(111) surface with p(2x2) adsorbate overlayer (1/4 coverage). 
Please use PBE pseudopotential and PBE exchange correlation function.
Literatures suggest that ontop site is 0.108 eV less stable than fcc site when using PBE xc. 
If your result is not within 10 percent of the literature, please find out possible reasons and resolve it."""

userMessage_11 = "I am trying to study adsorption of CO on Pt111 surface at fcc site. Job CO_Pt111_fcc_upright_k_0.3_ecutwfc_60.pwi did not converge, please figure out why and resolve the convergence issue."

userMessage_12 = """please find the adsorption energy difference between the most favorable configurations (different adsorbate orientations 0, 90, 180) at fcc site and most favorable configuration (different adsorbate orientations 0, 90, 180) at ontop site for CO on Pt(111) surface with p(2x2) adsorbate overlayer (1/4 coverage), and analyze the uncertainty.
Please use PBE pseudopotential and Bayesian Error Estimation Functional (BEEF) exchange correlation function.
Literatures suggest that ontop site is 0.18 eV less stable than fcc site when using PBE xc.
If your result is not within 10 percent of the literature, please find out possible reasons and resolve it."""

userMessage_13 = """please conduct a initial screening on the default dataset on potential candidates as a catalyst for OER reaction. Please only consider O only and skip the study of OH and OOH for now. Please save the cadidates dataframe into a csv file."""

userMessage_14 = """please conduct a OER screening study to find out the best system to use as catalyst for OER reaction. You must evaluate more less than 3 different systems. Please only consider O only and skip the study of OH and OOH for now. Available systems can be found in the default dataset. you must include Ir, with Nsite < 20. Please use VASP as the calculator. 
"""

testMessage = """
please conduct a acidic OER screening study to find out the best system to use as catalyst for OER reaction with limited computational budget.
You must decide a screening strategy for selecting candidates materials and performing relavent DFT calculation to evaluate the OER activity.
Please only consider O only and skip the study of OH and OOH for now. 
Available systems can be found in the default dataset. you must use Nsite < 20. Please use VASP as the calculator. 
"""

config = load_config(os.path.join('./config', "default.yaml"))
# check_config(config)

WORKING_DIRECTORY = var.my_WORKING_DIRECTORY
# print N number of '#', where n = len("##  Working directory: " + WORKING_DIRECTORY + " ##")
print("#" * (len("##  Working directory: " + WORKING_DIRECTORY + " ##")))
print("##  Working directory: " + WORKING_DIRECTORY + " ##")
print("#" * (len("##  Working directory: " + WORKING_DIRECTORY + " ##")))

assert WORKING_DIRECTORY is not None, "Please set the WORKING_DIRECTORY var"

CANVAS.set_working_directory(WORKING_DIRECTORY)
# CANVAS.canvas["finished_job_list"] = ["CO_Pt111_fcc_upright_k_0.3_ecutwfc_60.pwi"]

# set environment variable
os.environ["OMP_NUM_THREADS"] = "1"

# check if working directory exists, if so delete it
if os.path.exists(WORKING_DIRECTORY):
    os.system(f"rm -rf {WORKING_DIRECTORY}")

os.makedirs(WORKING_DIRECTORY, exist_ok=False)

EXPLOG.init(Path(WORKING_DIRECTORY)/"TEMP_vasp_calcs", "test")
# check if resource_suggestions.db exist in the working directory
db_file = os.path.join(WORKING_DIRECTORY, 'resource_suggestions.db')
if os.path.exists(db_file):
    os.remove(db_file)
initialize_database(db_file)

graph = create_planning_graph(config)
llm_config = {"thread_id": "1", 'recursion_limit': 1000}

# print(graph)


# save_graph_to_file(graph, WORKING_DIRECTORY, "super_graph")
# exit()


# for s in graph.stream(
# {
#     "messages": [
#         HumanMessage(content=f"{userMessage_4}")
#     ]
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")

# for s in graph.stream(
# {
#     "messages": [
#         HumanMessage(content=f"{userMessage_2}")
#     ]
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")

# for s in graph.stream(
# {
#     "input": f"{userMessage_6}",
#     "plan": [],
#     "past_steps": []
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")
        
# for s in graph.stream(
# {
#     "input": [
#         HumanMessage(content=f"{userMessage_5}")
#     ]
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")

print("Start, check the log file for details")
log_filename = f"./log/agent_stream_{int(time.time())}.log"  # Add timestamp to filename
with open(log_filename, "a") as log_file:
    log_file.write(f"=== Session started at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
    if eval(config["SAVE_DIALOGUE"]):
        with open(f"{WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"=== Session started at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
    
    for s in graph.stream(
        {
            "inputs": f"{testMessage}",
            "plan": [],
            "past_steps": []
        }, llm_config):
        
        if "__end__" not in s:
            print(s)
            print("----")
            if eval(config["SAVE_DIALOGUE"]):
                with open(f"{WORKING_DIRECTORY}/his.txt", "a") as f:
                    f.write(repr(s) + "\n")
                    f.write("----\n")
            
            # time.sleep(5)
            # Print to console
            log_file.write(f"{s}\n")
            log_file.write("----\n")
            log_file.flush()
    log_file.write(f"=== Session ended at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
    if eval(config["SAVE_DIALOGUE"]):
        with open(f"{WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"=== Session ended at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
print("End, check the log file for details")


Setting GPU_AVAILABLE == True
GPU_AVAILABLE: True
########################################################################################################
##  Working directory: /nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools_moreMoney ##
########################################################################################################
Start, check the log file for details
supervisor is processing!!!!!
{'inputs': '\nplease conduct a acidic OER screening study to find out the best system to use as catalyst for OER reaction with limited computational budget.\nYou must decide a screening strategy for selecting candidates materials and performing relavent DFT calculation to evaluate the OER activity.\nPlease only consider O only and skip the study of OH and OOH for now. \nAvailable systems can be found in the default dataset. you must use Nsite < 20. Please use VASP as the calculator. \n', 'plan': [], 'past_steps': []}


/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! tool_choice is not default parameter.
                tool_choice was transferred to model_kwargs.
                Please confirm that tool_choice is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


input_tokens: 1765, output_tokens: 14
================================== Ai Message ==================================

[{'id': 'toolu_0122Y3fvPRCEKZVHrwZ1UFbh', 'input': {}, 'name': 'inspect_my_canvas', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool Calls:
  inspect_my_canvas (toolu_0122Y3fvPRCEKZVHrwZ1UFbh)
 Call ID: toolu_0122Y3fvPRCEKZVHrwZ1UFbh
  Args:

================================= Tool Message =================================
Name: inspect_my_canvas

[]

input_tokens: 1829, output_tokens: 34
================================== Ai Message ==================================

[{'id': 'toolu_017shJ1wXpRcVzZCqoGWiY3F', 'input': {'only_get_updates': False}, 'name': 'inspect_explog', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool Calls:
  inspect_explog (toolu_017shJ1wXpRcVzZCqoGWiY3F)
 Call ID: toolu_017shJ1wXpRcVzZCqoGWiY3F
  Args:
    only_get_updates: False

================================= Tool Message =================================
Name: inspect_explog



100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 528971/528971 [01:18<00:00, 6701.59it/s]


Found 32 stable entries given the stability criteria.
##################### CANVAS #######################
{
'OER_Agent_Expertise': '## OER Agent Expertise and Capabilities\n\n### What I Can Do:\n1. **Data Analysis & Filtering**: Filter and sort materials database based on stability criteria (pH, electrochemical potential, decomposition threshold), structural properties, and composition\n2. **Candidate Selection**: Identify promising OER catalyst candidates from materials databases using various screening criteria\n3. **DFT Workflow Management**: \n   - Submit bulk relaxation calculations\n   - Submit surface/termination relaxation calculations\n   - Submit O and OH adsorption calculations (though OH is skipped per current objective)\n4. **Surface Analysis**: \n   - Rank surface terminations based on coordination analysis\n   - Identify and analyze adsorption sites (on-top and lattice O sites)\n5. **Progress Tracking**: Monitor calculation status and results through EXPLOG\n6. **Litera

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 528971/528971 [01:19<00:00, 6667.22it/s]


Found 28971 stable entries given the stability criteria.
##################### CANVAS #######################
{
'OER_Agent_Expertise': '## OER Agent Expertise and Capabilities\n\n### What I Can Do:\n1. **Data Analysis & Filtering**: Filter and sort materials database based on stability criteria (pH, electrochemical potential, decomposition threshold), structural properties, and composition\n2. **Candidate Selection**: Identify promising OER catalyst candidates from materials databases using various screening criteria\n3. **DFT Workflow Management**: \n   - Submit bulk relaxation calculations\n   - Submit surface/termination relaxation calculations\n   - Submit O and OH adsorption calculations (though OH is skipped per current objective)\n4. **Surface Analysis**: \n   - Rank surface terminations based on coordination analysis\n   - Identify and analyze adsorption sites (on-top and lattice O sites)\n5. **Progress Tracking**: Monitor calculation status and results through EXPLOG\n6. **Lit

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 528971/528971 [01:19<00:00, 6655.42it/s]


Found 28971 stable entries given the stability criteria.
##################### CANVAS #######################
{
'OER_Agent_Expertise': '## OER Agent Expertise and Capabilities\n\n### What I Can Do:\n1. **Data Analysis & Filtering**: Filter and sort materials database based on stability criteria (pH, electrochemical potential, decomposition threshold), structural properties, and composition\n2. **Candidate Selection**: Identify promising OER catalyst candidates from materials databases using various screening criteria\n3. **DFT Workflow Management**: \n   - Submit bulk relaxation calculations\n   - Submit surface/termination relaxation calculations\n   - Submit O and OH adsorption calculations (though OH is skipped per current objective)\n4. **Surface Analysis**: \n   - Rank surface terminations based on coordination analysis\n   - Identify and analyze adsorption sites (on-top and lattice O sites)\n5. **Progress Tracking**: Monitor calculation status and results through EXPLOG\n6. **Lit

In [7]:
study = EXPLOG.relational_frame.candidates['8a3e128c32'].study_obj

KeyError: "Table 'candidates' expected 1 row where candidate_id=='8a3e128c32', got 0"

In [8]:
FINAL_REPORT = "## Acidic OER Catalyst Screening Study - COMPLETED\n\n### Project Summary\n\nI have successfully completed the comprehensive acidic OER catalyst screening study with limited computational budget. The study systematically evaluated 15 candidate materials through DFT calculations using VASP, focusing on O-only adsorption analysis.\n\n---\n\n### TOP CATALYST RECOMMENDATIONS\n\n**RANK 1: EuNiIO6 (overpotential = 0.05V)**\n- Performance: 5-7x better than IrO2 benchmark\n- Advantages: Excellent cost-effectiveness, proven oxide stability, scalable synthesis\n- Mechanism: Ni eg1 orbital occupancy provides intrinsically favorable O binding\n- Recommendation: HIGHEST PRIORITY for experimental validation\n\n**RANK 2: EuAgF4 (overpotential = 0.03V)**\n- Performance: 10x better than IrO2 benchmark (best in study)\n- Advantages: Exceptional overpotential, novel fluoride chemistry\n- Concerns: Fluoride stability in acidic conditions needs validation\n- Recommendation: HIGH PRIORITY with stability testing\n\n**RANK 3: EuCuF6 (overpotential = 0.28V)**\n- Performance: Comparable to IrO2 at much lower cost\n- Advantages: Cost-effective noble-metal-free alternative\n- Recommendation: MEDIUM PRIORITY for cost-sensitive applications\n\n---\n\n### KEY SCIENTIFIC DISCOVERIES\n\n1. Fluoride Ligand Effect: Fluoride ligands dramatically enhance OER activity (16x improvement for Ag, 1.7x for Cu) by withdrawing electron density and tuning O binding to optimal values\n\n2. Noble Metal Paradox: Noble metals (Ir, Pt, Au) in oxide frameworks underperform expectations. MnIr3O8 showed worst performance (overpotential=1.20V) due to O overbinding\n\n3. Bulk Electronic Structure Dominance: Surface termination has minimal effect - bulk electronic properties dominate catalytic activity\n\n4. Volcano Plot Validation: Clear correlation with optimal Gibbs free energy = 2.46 +/- 0.1 eV\n\n---\n\n### Screening Campaign Statistics\n\n- Initial dataset: 2,557 Pourbaix-stable materials (pH 0-1, U=1.23-2.0V)\n- Candidates selected: 15 diverse materials (6-12 atoms, 7 transition metals + 4 noble metals)\n- Bulk calculations: 15/15 completed (100% success)\n- Surface calculations: 16 terminations across 8 top candidates (100% success)\n- O adsorption calculations: 19 sites analyzed (17 on-top metal + 5 lattice O sites, 100% success)\n- Computational efficiency: Achieved comprehensive screening within budget constraints\n\n---\n\n### Design Principles Established\n\n1. Combine weak O-binding metals (Ag, Cu) with strong electron-withdrawing ligands (F-)\n2. Target metallic/near-metallic materials (bandgap < 0.01 eV)\n3. Optimize Gibbs free energy to 2.46 +/- 0.1 eV for minimal overpotential\n4. Prioritize fluoride > sulfate > oxide ligand environments\n5. Consider Ni-based oxides for balanced performance and practicality\n\n---\n\n### Future Work Recommendations\n\n1. CRITICAL: Validate OH* and OOH* intermediates for complete 4-electron mechanism\n2. HIGH PRIORITY: Test fluoride material stability in acidic electrolytes\n3. RECOMMENDED: Perform electronic structure analysis (DOS, d-band center, Bader charges)\n4. SUGGESTED: Include solvation effects and kinetic barrier calculations\n5. EXPLORATORY: Expand fluoride-based material screening (underexplored chemical space)\n\n---\n\nThe full comprehensive report with all technical details, analysis methodology, energetic data, structure-activity relationships, literature comparisons, and practical considerations has been documented on CANVAS under key 'FINAL_COMPREHENSIVE_OER_REPORT'.\n\nThe screening study is now complete. All objectives have been achieved within the computational budget constraints."
print(FINAL_REPORT)

## Acidic OER Catalyst Screening Study - COMPLETED

### Project Summary

I have successfully completed the comprehensive acidic OER catalyst screening study with limited computational budget. The study systematically evaluated 15 candidate materials through DFT calculations using VASP, focusing on O-only adsorption analysis.

---

### TOP CATALYST RECOMMENDATIONS

**RANK 1: EuNiIO6 (overpotential = 0.05V)**
- Performance: 5-7x better than IrO2 benchmark
- Advantages: Excellent cost-effectiveness, proven oxide stability, scalable synthesis
- Mechanism: Ni eg1 orbital occupancy provides intrinsically favorable O binding
- Recommendation: HIGHEST PRIORITY for experimental validation

**RANK 2: EuAgF4 (overpotential = 0.03V)**
- Performance: 10x better than IrO2 benchmark (best in study)
- Advantages: Exceptional overpotential, novel fluoride chemistry
- Concerns: Fluoride stability in acidic conditions needs validation
- Recommendation: HIGH PRIORITY with stability testing

**RANK 3: EuCu

In [15]:
candidate_df = EXPLOG.relational_frame.candidates.df
candidate_df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,f99f34ee94,Noble metal oxide IrPdO4 with Ir (known OER ch...,Rank 1 candidate - highest priority due to Ir ...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
1,2b3cd20212,Co-based oxide CoHgTeO6 with Co (literature-pr...,Rank 2 candidate - Co-based oxide with excelle...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
2,2787f43669,Ni-based oxide NiPbIO6 with Ni (proven in Co-N...,Rank 3 candidate - Ni-based oxide with extreme...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
3,5e5c060528,Ag-based oxide KAg(HgO2)2 with Ag (literature-...,Rank 4 candidate - Ag-based oxide with high co...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
4,2730bcba45,Co-based oxide InCoAsO6 with Co and As. Metall...,Rank 5 candidate - Co-based oxide with As for ...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
5,0b3b975fa9,Cr-based oxide CrCdIO6 with Cr (literature-exp...,Rank 6 candidate - Cr-based oxide for element ...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
6,c5524fdca6,Cu-based fluoride EuCuF6 with Cu (literature-e...,Rank 7 candidate - Cu-based fluoride for struc...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.28
7,5925f0bb17,Mn-based oxide EuMnIO6 with Mn (literature-pro...,Rank 8 candidate - Mn-based oxide with multipl...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.58
8,728d6560b8,Ni-based oxide EuNiIO6 with Ni. Near-metallic ...,Rank 9 candidate - Ni-based oxide for Co-Ni co...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.05
9,caa0d62544,Sb-Pd oxide EuSbPdO6 with Sb (literature-ident...,Rank 10 candidate - Sb-Pd oxide with p-d hybri...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN


In [16]:
process_df = EXPLOG.relational_frame.processes.df
process_df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,f99f34ee94,bulk_relaxation,42,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for IrPdO4 - Rank 1 noble meta...
1,1,2b3cd20212,bulk_relaxation,43,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for CoHgTeO6 - Rank 2 Co-based...
2,2,2787f43669,bulk_relaxation,44,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for NiPbIO6 - Rank 3 Ni-based ...
3,3,5e5c060528,bulk_relaxation,45,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for KAg(HgO2)2 - Rank 4 Ag-bas...
4,4,2730bcba45,bulk_relaxation,46,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for InCoAsO6 - Rank 5 Co-based...
5,5,0b3b975fa9,bulk_relaxation,47,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for CrCdIO6 - Rank 6 Cr-based ...
6,6,c5524fdca6,bulk_relaxation,48,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for EuCuF6 - Rank 7 Cu-based f...
7,7,5925f0bb17,bulk_relaxation,49,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for EuMnIO6 - Rank 8 Mn-based ...
8,8,728d6560b8,bulk_relaxation,50,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for EuNiIO6 - Rank 9 Ni-based ...
9,9,caa0d62544,bulk_relaxation,51,completed,None,None,/nfs/turbo/coe-venkvis/ziqiw-turbo/material_ag...,Bulk relaxation for EuSbPdO6 - Rank 10 Sb-Pd o...


In [17]:
my_EXPLOG = {"candidate_df": candidate_df, "process_df": process_df}

In [18]:
with open("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools_moreMoney/EXPLOG.pickle", "wb") as f:
    pickle.dump(my_EXPLOG, f)

In [19]:
with open("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools_moreMoney/EXPLOG.pickle", "rb") as f:
    asdhfabvd = pickle.load(f)

In [9]:
EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,f99f34ee94,Noble metal oxide IrPdO4 with Ir (known OER ch...,Rank 1 candidate - highest priority due to Ir ...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
1,2b3cd20212,Co-based oxide CoHgTeO6 with Co (literature-pr...,Rank 2 candidate - Co-based oxide with excelle...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
2,2787f43669,Ni-based oxide NiPbIO6 with Ni (proven in Co-N...,Rank 3 candidate - Ni-based oxide with extreme...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
3,5e5c060528,Ag-based oxide KAg(HgO2)2 with Ag (literature-...,Rank 4 candidate - Ag-based oxide with high co...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
4,2730bcba45,Co-based oxide InCoAsO6 with Co and As. Metall...,Rank 5 candidate - Co-based oxide with As for ...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
5,0b3b975fa9,Cr-based oxide CrCdIO6 with Cr (literature-exp...,Rank 6 candidate - Cr-based oxide for element ...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
6,c5524fdca6,Cu-based fluoride EuCuF6 with Cu (literature-e...,Rank 7 candidate - Cu-based fluoride for struc...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.28
7,5925f0bb17,Mn-based oxide EuMnIO6 with Mn (literature-pro...,Rank 8 candidate - Mn-based oxide with multipl...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.58
8,728d6560b8,Ni-based oxide EuNiIO6 with Ni. Near-metallic ...,Rank 9 candidate - Ni-based oxide for Co-Ni co...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.05
9,caa0d62544,Sb-Pd oxide EuSbPdO6 with Sb (literature-ident...,Rank 10 candidate - Sb-Pd oxide with p-d hybri...,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN


In [10]:
study = EXPLOG.relational_frame.candidates["9d5fbcd27b"].study_obj
surface_study_dict = study.get_surface_studies()
surface_study_keys = list(surface_study_dict.keys())
surface_study = surface_study_dict[surface_study_keys[0]]
surface_study.get_adsorption_sites_df()

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH),ideal overpotential
0,0,on-top,Mn,"[(O, 1.9746988832513774), (O, 1.97469888325137...",N/A,"[-1.2520215275045938, 4.738826073310954, 25.27...",3.0,None,None,None,None,None,None,None
1,1,on-top,Ir,"[(O, 1.9495696331772732), (O, 2.00939021306152...",N/A,"[2.2757943329601638, 4.7406202307547325, 24.66...",13.0,4.856318,None,None,None,None,None,1.2
2,2,on-top,Ir,"[(O, 1.9853385426656671), (O, 1.98533854266566...",N/A,"[0.19914866172460058, 1.8937800471067303, 25.2...",14.0,None,None,None,None,None,None,None
3,3,on-top,Ir,"[(O, 1.949735653994512), (O, 2.009627472619713...",N/A,"[3.7234580189202204, 1.8937800471067319, 24.66...",15.0,None,None,None,None,None,None,None
4,4,lattice O,N/A,"[(Ir, 1.9883167779478808), (Mn, 2.097046927397...",1,"[-0.6312240640708017, 3.1845653895784554, 24.5...",44.0,-0.21654,None,None,None,None,None,1.34
5,5,lattice O,N/A,"[(Ir, 1.9896387113290135), (Mn, 2.095541135819...",1,"[0.7542545553061949, 0.4601040330090424, 24.51...",45.0,None,None,None,None,None,None,None


In [37]:
from ase.visualize import view
bulk = study.get_bulk_atoms()

surface_study_dict = study.get_surface_studies()
surface_study_keys = list(surface_study_dict.keys())
surface_study = surface_study_dict[surface_study_keys[0]]
surface_atoms = surface_study.get_relaxed_surface()

ad_site_study_dict = surface_study.get_adsorption_site_studies_dict()
ad_site_study_keys = list(ad_site_study_dict.keys())
ad_site_study = ad_site_study_dict[ad_site_study_keys[0]]
ad_site_atoms = ad_site_study.relaxed_O_atoms


view(ad_site_atoms, viewer="x3d", color = 'tags')


In [39]:
surface_study.get_adsorption_sites_df()

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH),ideal overpotential
0,0,on-top,Mn,"[(O, 1.9379731233488842), (O, 1.93831816738117...",N/A,"[3.7527208194047312, 0.8422374340513212, 30.46...",6.0,2.450849,None,None,None,None,None,0.0
1,1,on-top,Sb,"[(O, 2.008608544771258), (O, 2.009084078481537...",N/A,"[-1.404512924005649, 3.828260051784206, 30.478...",10.0,2.987866,None,None,None,None,None,0.26
2,2,lattice O,N/A,"[(Mn, 1.9383181673811793), (Sb, 2.009522248862...",1,"[0.17467952849407345, 0.7052370162327237, 29.5...",29.0,1.956699,None,None,None,None,None,0.25
3,3,lattice O,N/A,"[(Mn, 1.9394476702120593), (Sb, 2.009678725547...",1,"[0.2623561129003086, 4.005607068355536, 29.515...",30.0,None,None,None,None,None,None,None
4,4,lattice O,N/A,"[(Mn, 1.9385330347038172), (Sb, 2.009084078481...",1,"[3.072008398881783, 2.284875023789384, 29.5086...",31.0,None,None,None,None,None,None,None


In [41]:
EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,8a3e128c32,EuMnSbO6: Mn-based trigonal oxide with excelle...,First candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.0
1,5925f0bb17,"EuMnIO6: Mn-based trigonal oxide with iodine, ...",Second candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.22
2,0852585158,MnAg(SO4)2: Mn-sulfate with monoclinic structu...,Third candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.02
3,728d6560b8,"EuNiIO6: Ni-based trigonal oxide with iodine, ...",Fourth candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.65
4,d9a54f6d10,"LiNi3O8: Li-Ni oxide with trigonal structure, ...",Fifth candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.51
5,c2e2511a3d,BaNiIrO6: Ni-Ir double perovskite with trigona...,Sixth candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.32
6,6d230e06ac,NiRh3O8: Ni-Rh oxide with monoclinic structure...,Seventh candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.22
7,db01847f5f,MnAg(SeO4)2: Mn-selenate with monoclinic struc...,Eighth candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.64
8,2b3cd20212,CoHgTeO6: Co-based trigonal oxide with Hg and ...,Ninth candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.01
9,50b7383592,Co(TcO4)2: Co-based trigonal oxide with techne...,Tenth candidate from top 10 diverse selection,<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,0.04


In [27]:
surface_study = list(surface_study_dict.keys())
# keys = []
# for k in surface_study_dict.keys():
#     keys.append(k)
# keys
surface_study

[3, 2]

In [7]:
def submit_dft_job(
    MaterialId: Annotated[str, "MaterialId of the candidate to submit DFT job for."],
    calculation_type: Annotated[Literal['bulk_relaxation', 'surface_relaxation', 'OH_adsorption', 'O_adsorption'], "Type of DFT calculation to submit."],
    note: Annotated[str, "Short note you want to leave for the calculation"],
    termination_index: Annotated[int, "termination index. Only needed for surface and adsorption calculations"] = None,
    ad_site_index: Annotated[int, "index of the site you want to adsorb O or OH onto. Only neeeded for adsorption calculations"] = None,
    partition: Annotated[Literal['xeon56', 'xeon40el8', 'xeon24el8', 'auto'], "Partition to submit the job to"] = "auto",
):
    """Submit different types of DFT jobs to the cluster for a cadidate"""

    # --- Sanity checks for input arguments ----------------------------
    if ad_site_index is not None and termination_index is None:
        raise ValueError("termination_index must be provided for" \
        " adsorption calculations")
    
    # MORE CHECKS NEEDED...!!!

    # ------------------------------------------------------------------
    
    # --- Initializing surface- and adsorption studies if not 
    # already initialized ----------------------------------------------
    if termination_index is not None:
        study = EXPLOG.relational_frame.candidates[MaterialId].study_obj

        surface_study_dict = study.get_surface_studies()
        if termination_index not in surface_study_dict.keys():
            study.initialize_oer_surface_study(termination_index)
        surface_study = study.get_surface_studies()[termination_index]

        if ad_site_index is not None:
            ad_site_studies_dict = surface_study.get_adsorption_site_studies_dict()
            if ad_site_index not in ad_site_studies_dict.keys():
                surface_study.initialize_adsorption_site_study(ad_site_index)
            ad_site_study = surface_study.get_adsorption_site_studies_dict()[ad_site_index]
    # ------------------------------------------------------------------
            
    id = EXPLOG.add_process(MaterialId, calculation_type, termination_index, ad_site_index, note)
    EXPLOG.submit_process(id, partition)
    # save EXPLOG into a pickle file under WORKING_DIRECTORY for record and future reference
    # with open(os.path.join(var.my_WORKING_DIRECTORY, "EXPLOG.pkl"), "wb") as f:
    #     pickle.dump(EXPLOG, f)
    
    return f"Submitted {calculation_type} for candidate {MaterialId}"


In [ ]:
submit_dft_job(
    MaterialId="129fb15eca",
    calculation_type="surface_relaxation",
    termination_index=1,
    note="note"
)

UnboundLocalError: cannot access local variable 'surface_study' where it is not associated with a value

In [ ]:
with open("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools/canvas.pickle", "rb") as f:
    lastCanvas = pickle.load(f)

In [8]:
lastCanvas["OER_stable_entries_df"]

,Composition,MaterialId,Reduced Formula,Elements,NSites,Crystal System,Dimensionality Cheon,Bandgap,Disorder Probability,average_HHI_P,average_HHI_P_excluding_OHCNPS,max_HHI_P,average_HHI_R,average_HHI_R_excluding_OHCNPS,max_HHI_R
298967,Eu1Mn1O6Sb1,8a3e128c32,EuMnSbO6,"[O, Mn, Sb, Eu]",9,trigonal,3D,0.0006,0.78893,2444,6333,9500,1256,2767,3400


In [9]:
var.my_RESOURCE_DIRECTORY

{}

In [7]:
EXPLOG.init(Path(WORKING_DIRECTORY)/"TEMP_vasp_calcs_testtest", "test")

In [8]:
EXPLOG.update_log()

{}

In [7]:
with open("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools/TEMP_vasp_calcs/explog.pkl", "rb") as f:
    tmpexplog = pickle.load(f)

RecursionError: maximum recursion depth exceeded

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,129caa3e7d,Co-based perovskite-like structure (LaCoSbO6) ...,"NSites=9, trigonal, 3D, Bandgap=1.33 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
1,2730bcba45,Co-based oxide (InCoAsO6) with low disorder pr...,"NSites=9, trigonal, 3D, Bandgap=0.51 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
2,859bfbde9f,Ce-Co oxide (CeCoAsO6) with moderate bandgap (...,"NSites=9, trigonal, 3D, Bandgap=1.17 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
3,8b19878c1f,"Ce-Co phosphate (CeCo2PO8) with two Co atoms, ...","NSites=12, triclinic, 3D, Bandgap=0.15 eV, Dis...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
4,c2e2511a3d,Ba-Ni-Ir oxide (BaNiIrO6) with near-metallic b...,"NSites=9, trigonal, Bandgap=0.003 eV, Disorder...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
5,8668c079e1,Sc-Ni-Bi oxide (ScNi2BiO6) with two Ni atoms a...,"NSites=10, trigonal, 3D, Bandgap=0.82 eV, Diso...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
6,4d1be4578b,"Ni-Ir oxide (NiIr3O8) with three Ir atoms, mon...","NSites=12, monoclinic, 3D, Disorder=0.15",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
7,d9a54f6d10,"Li-Ni oxide (LiNi3O8) with three Ni atoms, nea...","NSites=12, trigonal, Bandgap=0.002 eV, Disorde...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
8,141a2241cc,Sr-Mn-W oxide (SrMnWO6) with trigonal structur...,"NSites=9, trigonal, Bandgap=2.17 eV, Disorder=...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
9,9272d7a1fe,Nd-Mn-Pt oxide (NdMnPtO6) with near-metallic b...,"NSites=9, trigonal, 3D, Bandgap=0.01 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN


In [12]:
id = EXPLOG.add_process("8b19878c1f", "bulk_relaxtion")

In [13]:
id

'0'